In [2]:
"""
NGS allele reversion analysis for exon 5

This script combines CRISPResso-style allele count tables, translates edited alleles,
classifies protein-level outcomes, summarizes allele classes across conditions, and
plots log2 fold-change relative to day 7.

Input files are expected to be tab-delimited .txt files containing at least:
    Aligned_Sequence, Reference_Sequence, #Reads

Sample names should include condition and replicate labels such as:
    day_7_a, day_7_b, day_7_c, day_14_a, mock_a, nira_a, etc.
"""

from __future__ import annotations

import itertools
import os
import re
from functools import reduce
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from Bio.Seq import Seq


# -----------------------------------------------------------------------------
# User settings
# -----------------------------------------------------------------------------

INPUT_DIR = Path("/path/to/sample_input")
OUTPUT_DIR = Path("/path/to/sample_output")

CONDITIONS = ["day_7", "day_14", "mock", "nira"]
REPLICATES = ["a", "b", "c"]

START_CODON = "TTTAGT"
STOP_CODON = "TCAGGT"
WT_PROTEIN = "PVVLQCTHVTPQRDKS"

FIVE_PRIME_TRIM = "FS"
THREE_PRIME_TRIM = "GM"

MUTATION_ORDER = ["Wild-type", "in-frame", "Reversion", "Frame-shift"]
PLOT_CONDITIONS = ["Day 14", "Mock", "1uM Nira"]
PLOT_COND_KEYS = ["day_14_log2fc", "mock_log2fc", "1um_nira_log2fc"]
PLOT_REP_PREFIX = {
    "Day 14": "day_14",
    "Mock": "mock",
    "1uM Nira": "nira",
}

#adjust as needed
PSEUDOCOUNT = 0


# -----------------------------------------------------------------------------
# Sequence utilities
# -----------------------------------------------------------------------------

def reverse_complement(seq: str) -> str:
    """Return the reverse complement of a DNA sequence."""
    complement = str.maketrans("ACGTNacgtn", "TGCANtgcan")
    return str(seq).translate(complement)[::-1]


def translate_sequence(sequence: str, start_codon: str, stop_codon: str) -> str:
    """Translate the region between the specified start and stop codons."""
    sequence = str(sequence).upper().replace("-", "")
    start_codon = start_codon.upper()
    stop_codon = stop_codon.upper()

    start_idx = sequence.find(start_codon)
    stop_idx = sequence.find(stop_codon)

    if start_idx == -1 or stop_idx == -1 or stop_idx < start_idx:
        return ""

    seq_to_translate = sequence[start_idx : stop_idx + len(stop_codon)]
    return str(Seq(seq_to_translate).translate(table="Standard", to_stop=True))


def add_translation_column(
    df: pd.DataFrame,
    start_codon: str = START_CODON,
    stop_codon: str = STOP_CODON,
    sequence_col: str = "aligned_sequence",
) -> pd.DataFrame:
    """Add a protein translation column to an allele table."""
    df = df.copy()
    df["Translation"] = df[sequence_col].apply(
        lambda seq: translate_sequence(seq, start_codon, stop_codon)
    )
    return df


def trim_translation_endings(
    df: pd.DataFrame,
    translation_col: str = "Translation",
    five_prime_trim: str = FIVE_PRIME_TRIM,
    three_prime_trim: str = THREE_PRIME_TRIM,
) -> pd.DataFrame:
    """Remove expected cloning/amplicon-derived residues from translation ends."""
    df = df.copy()

    def trim_sequence(seq: object) -> object:
        if not isinstance(seq, str):
            return seq

        if seq.startswith(five_prime_trim):
            seq = seq[len(five_prime_trim) :]

        if seq.endswith(three_prime_trim):
            seq = seq[: -len(three_prime_trim)]
        elif seq.endswith(("G", "M")):
            seq = seq[:-1]

        return seq

    df[translation_col] = df[translation_col].apply(trim_sequence)
    return df


def mark_putative_frameshifts(
    df: pd.DataFrame,
    translation_col: str = "Translation",
    terminal_residue: str = "S",
) -> pd.DataFrame:
    """Append '*' to translations that do not end with the expected terminal residue."""
    df = df.copy()
    translations = df[translation_col].fillna("").astype(str)

    mask = ~translations.str.endswith(terminal_residue) & ~translations.str.endswith("*")
    df.loc[mask, translation_col] = translations.loc[mask] + "*"

    return df


# -----------------------------------------------------------------------------
# Input and read normalization
# -----------------------------------------------------------------------------

def read_allele_file(file_path: Path) -> pd.DataFrame | None:
    """Read one allele count table and label read-count columns by file name."""
    try:
        df = pd.read_csv(file_path, sep="\t")
    except Exception as exc:
        print(f"Skipping {file_path.name}: could not read file ({exc})")
        return None

    if "Aligned_Sequence" not in df.columns:
        print(f"Skipping {file_path.name}: missing Aligned_Sequence column")
        return None

    sample_name = file_path.stem
    rename_dict = {}

    if "#Reads" in df.columns:
        rename_dict["#Reads"] = f"{sample_name}_#Reads"
    if "%Reads" in df.columns:
        rename_dict["%Reads"] = f"{sample_name}_%Reads"

    df = df.rename(columns=rename_dict)
    df["Reverse_Complement"] = df["Aligned_Sequence"].apply(reverse_complement)

    merge_cols = [
        "Aligned_Sequence",
        "Reference_Sequence",
        "Reverse_Complement",
        "n_inserted",
        "n_deleted",
        "n_mutated",
    ]
    cols_to_keep = [col for col in merge_cols + list(rename_dict.values()) if col in df.columns]

    return df[cols_to_keep]


def merge_allele_files(input_dir: Path) -> pd.DataFrame:
    """Merge all tab-delimited allele count files in an input directory."""
    files = sorted(input_dir.glob("*.txt"))
    dfs = [read_allele_file(file_path) for file_path in files]
    dfs = [df for df in dfs if df is not None]

    if not dfs:
        raise FileNotFoundError(f"No valid .txt allele files found in {input_dir}")

    merge_cols = [
        "Aligned_Sequence",
        "Reference_Sequence",
        "Reverse_Complement",
        "n_inserted",
        "n_deleted",
        "n_mutated",
    ]
    merge_cols = [col for col in merge_cols if all(col in df.columns for df in dfs)]

    merged = reduce(
        lambda left, right: pd.merge(left, right, on=merge_cols, how="outer"),
        dfs,
    ).fillna(0)

    read_cols = [col for col in merged.columns if col.endswith("_#Reads")]
    merged["common_count"] = merged[read_cols].gt(0).sum(axis=1)
    merged["total_reads"] = merged[read_cols].sum(axis=1)

    merged = merged.sort_values(
        by=["common_count", "total_reads"],
        ascending=False,
    )

    merged = merged.drop(
        columns=["common_count", "total_reads", "Reverse_Complement"],
        errors="ignore",
    )

    merged.columns = merged.columns.str.lower()

    percent_cols = [col for col in merged.columns if "%" in col]
    return merged.drop(columns=percent_cols, errors="ignore")


def combine_replicate_read_counts(
    df: pd.DataFrame,
    conditions: list[str] = CONDITIONS,
    replicates: list[str] = REPLICATES,
) -> pd.DataFrame:
    """Combine technical read-count columns and convert each sample to percent reads."""
    df = df.copy()

    for condition, replicate in itertools.product(conditions, replicates):
        sample_key = f"{condition}_{replicate}"
        matching_cols = [col for col in df.columns if re.search(sample_key, col)]

        if not matching_cols:
            continue

        sum_col = f"{sample_key}_read_sum"
        pct_col = f"{sample_key}%reads"

        df[sum_col] = df[matching_cols].sum(axis=1)
        total_reads = df[sum_col].sum()
        df[pct_col] = (df[sum_col] / total_reads) * 100 if total_reads else 0

    read_count_cols = [col for col in df.columns if "#reads" in col or col.endswith("_read_sum")]
    return df.drop(columns=read_count_cols, errors="ignore")


# -----------------------------------------------------------------------------
# Allele classification and summaries
# -----------------------------------------------------------------------------

def sum_reads_by_translation(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse identical protein translations by summing percent reads."""
    read_cols = [col for col in df.columns if col.endswith("%reads")]
    return df.groupby("Translation", as_index=False)[read_cols].sum()


def align_by_shared_ends(wt_seq: str, mut_seq: str) -> tuple[str, str]:
    """Align two short protein sequences by preserving common prefix and suffix."""
    start = 0
    while start < min(len(wt_seq), len(mut_seq)) and wt_seq[start] == mut_seq[start]:
        start += 1

    end = 0
    while (
        end < len(wt_seq) - start
        and end < len(mut_seq) - start
        and wt_seq[-(end + 1)] == mut_seq[-(end + 1)]
    ):
        end += 1

    wt_mid = wt_seq[start : len(wt_seq) - end]
    mut_mid = mut_seq[start : len(mut_seq) - end]

    if len(wt_mid) > len(mut_mid):
        mut_mid += "-" * (len(wt_mid) - len(mut_mid))
    elif len(mut_mid) > len(wt_mid):
        wt_mid += "-" * (len(mut_mid) - len(wt_mid))

    aligned_wt = wt_seq[:start] + wt_mid + wt_seq[len(wt_seq) - end :]
    aligned_mut = mut_seq[:start] + mut_mid + mut_seq[len(mut_seq) - end :]

    return aligned_wt, aligned_mut


def classify_translation(translation: str, wt_protein: str = WT_PROTEIN) -> str:
    """Assign a protein translation to a mutation class."""
    if not translation:
        return "Other"

    if translation.endswith("*"):
        return "Frame-shift"

    if translation == wt_protein:
        return "Wild-type"

    if len(translation) == len(wt_protein):
        mismatches = sum(a != b for a, b in zip(translation, wt_protein))
        return "Missense" if mismatches < 2 else "seq_error"

    aligned_wt, aligned_mut = align_by_shared_ends(wt_protein, translation)
    deletions = sum(a != "-" and b == "-" for a, b in zip(aligned_wt, aligned_mut))
    insertions = sum(a == "-" and b != "-" for a, b in zip(aligned_wt, aligned_mut))
    substitutions = sum(
        a != "-" and b != "-" and a != b
        for a, b in zip(aligned_wt, aligned_mut)
    )

    if deletions > 0 and insertions == 0 and substitutions == 0 and translation.endswith("S"):
        return "In_frame_deletion"

    if deletions > 0 and insertions == 0 and substitutions < 3 and translation.endswith("S"):
        return "Complex_in-frame"

    if translation.endswith("S") and len(translation) <= len(wt_protein) and "THV" not in translation:
        return "Reversion"

    return "Complex"


def classify_translations(df: pd.DataFrame) -> pd.DataFrame:
    """Trim translations, flag frameshifts, and add mutation class labels."""
    df = trim_translation_endings(df)
    df = mark_putative_frameshifts(df)
    df["type"] = df["Translation"].fillna("").astype(str).apply(classify_translation)
    return df


def summarize_by_type(df: pd.DataFrame) -> pd.DataFrame:
    """Collapse detailed mutation classes into plotting categories."""
    read_cols = [col for col in df.columns if col.endswith("%reads")]

    summary = df.groupby("type", as_index=False)[read_cols].sum()
    category_map = {
        "Missense": "in-frame",
        "In_frame_deletion": "in-frame",
        "Complex_in-frame": "in-frame",
        "Complex": "other",
        "seq_error": "other",
    }

    summary["type"] = summary["type"].replace(category_map)
    return summary.groupby("type", as_index=False)[read_cols].sum()


def add_condition_stats(
    df: pd.DataFrame,
    conditions: dict[str, list[str]],
) -> pd.DataFrame:
    """Add mean and standard deviation of percent reads for each condition."""
    stats_df = df.copy()

    for condition, columns in conditions.items():
        missing = [col for col in columns if col not in df.columns]
        if missing:
            raise KeyError(f"Missing columns for {condition}: {missing}")

        stats_df[f"{condition}_mean"] = df[columns].mean(axis=1)
        stats_df[f"{condition}_std"] = df[columns].std(axis=1)

    return stats_df


def add_log2_fold_changes(
    df: pd.DataFrame,
    reference_condition: str = "day_7",
    comparison_conditions: tuple[str, ...] = ("day_14", "mock", "nira"),
    replicates: list[str] = REPLICATES,
    pseudocount: float = PSEUDOCOUNT,
) -> pd.DataFrame:
    """Calculate replicate-level and mean log2 fold-change relative to a reference."""
    df = df.copy()
    df.columns = df.columns.str.lower()

    for condition in comparison_conditions:
        for replicate in replicates:
            current_col = f"{condition}_{replicate}%reads"
            reference_col = f"{reference_condition}_{replicate}%reads"
            log2fc_col = f"{condition}_{replicate}_log2fc"

            df[log2fc_col] = np.log2(
                (df[current_col] + pseudocount) / (df[reference_col] + pseudocount)
            )

    for condition in comparison_conditions:
        replicate_cols = [f"{condition}_{replicate}_log2fc" for replicate in replicates]
        df[f"{condition}_mean_log2fc"] = df[replicate_cols].mean(axis=1)

    return df.rename(
        columns={
            "day_14_mean_log2fc": "day_14_log2fc",
            "mock_mean_log2fc": "mock_log2fc",
            "nira_mean_log2fc": "1um_nira_log2fc",
        }
    )


# -----------------------------------------------------------------------------
# Plotting
# -----------------------------------------------------------------------------

def get_replicates_for_condition(
    df: pd.DataFrame,
    mutation_type: str,
    condition: str,
    rep_prefix: dict[str, str] = PLOT_REP_PREFIX,
    replicates: list[str] = REPLICATES,
) -> np.ndarray:
    """Return finite replicate log2FC values for one mutation type and condition."""
    cols = [f"{rep_prefix[condition]}_{replicate}_log2fc" for replicate in replicates]
    cols = [col for col in cols if col in df.columns]

    if not cols:
        return np.array([])

    values = df.loc[df["type"] == mutation_type, cols].to_numpy().flatten()
    return values[np.isfinite(values)]


def get_bar_height(df: pd.DataFrame, mutation_type: str, condition_key: str) -> float:
    """Return the mean log2FC bar height for one mutation type and condition."""
    if condition_key not in df.columns:
        return np.nan

    values = df.loc[df["type"] == mutation_type, condition_key].to_numpy().flatten()
    return float(values[0]) if len(values) else np.nan


def welch_pvalue(a: np.ndarray, b: np.ndarray, n_perm: int = 20_000) -> float:
    """Return Welch's t-test p-value, with a permutation fallback if SciPy is absent."""
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)
    a = a[np.isfinite(a)]
    b = b[np.isfinite(b)]

    if a.size < 2 or b.size < 2:
        return np.nan

    try:
        from scipy.stats import ttest_ind

        return float(ttest_ind(a, b, equal_var=False, nan_policy="omit").pvalue)
    except Exception:
        rng = np.random.default_rng(0)
        observed = abs(a.mean() - b.mean())
        pooled = np.concatenate([a, b])
        n_a = a.size
        count = 0

        for _ in range(n_perm):
            rng.shuffle(pooled)
            difference = abs(pooled[:n_a].mean() - pooled[n_a:].mean())
            count += difference >= observed

        return (count + 1) / (n_perm + 1)


def pvalue_to_stars(pvalue: float) -> str:
    """Convert a p-value to a conventional significance label."""
    if not np.isfinite(pvalue):
        return "n/a"
    if pvalue < 0.001:
        return "***"
    if pvalue < 0.01:
        return "**"
    if pvalue < 0.05:
        return "*"
    return "ns"


def draw_significance_bar(
    ax: plt.Axes,
    x1: float,
    x2: float,
    y: float,
    h: float,
    pvalue: float,
    fontsize: int = 11,
) -> None:
    """Draw one significance bracket and label."""
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=1.5, c="black")
    ax.text(
        (x1 + x2) / 2,
        y + h * 1.05,
        pvalue_to_stars(pvalue),
        ha="center",
        va="bottom",
        fontsize=fontsize,
        fontweight="bold",
    )


def plot_log2fc(
    df_plot: pd.DataFrame,
    output_path: Path,
    title: str = "Log2FC - exon 5",
) -> None:
    """Plot class-level mean log2FC with replicate points and WT comparisons."""
    df_plot = df_plot.copy()
    df_plot.columns = df_plot.columns.str.lower()
    df_plot = df_plot[df_plot["type"].isin(MUTATION_ORDER)]

    df_plot["type"] = pd.Categorical(
        df_plot["type"],
        categories=MUTATION_ORDER,
        ordered=True,
    )
    df_plot = df_plot.sort_values("type")

    colors = ["deepskyblue", "limegreen", "goldenrod", "deeppink"]
    x = np.arange(len(PLOT_CONDITIONS))
    bar_width = 0.8 / len(MUTATION_ORDER)
    offsets = np.linspace(
        -0.4 + bar_width / 2,
        0.4 - bar_width / 2,
        len(MUTATION_ORDER),
    )

    fig, ax = plt.subplots(figsize=(6, 6))

    for i, mutation_type in enumerate(MUTATION_ORDER):
        xi = x + offsets[i]
        heights = [get_bar_height(df_plot, mutation_type, key) for key in PLOT_COND_KEYS]

        ax.bar(
            xi,
            heights,
            width=bar_width,
            label=mutation_type,
            color=colors[i],
            edgecolor="black",
        )

        for j, condition in enumerate(PLOT_CONDITIONS):
            reps = get_replicates_for_condition(df_plot, mutation_type, condition)
            jitter = (np.random.rand(len(reps)) - 0.5) * bar_width * 0.7
            ax.scatter(
                xi[j] + jitter,
                reps,
                s=30,
                edgecolors="black",
                color=colors[i],
                zorder=3,
            )

    ax.set_xticks(x)
    ax.set_xticklabels(PLOT_CONDITIONS, fontsize=14, fontweight="bold", rotation=45)
    ax.set_ylabel("log$_2$ Fold Change", fontsize=14, fontweight="bold")
    ax.set_title(title, fontsize=18, fontweight="bold", pad=12)
    ax.set_ylim(-7, 10)
    ax.tick_params(axis="y", labelsize=14)

    for label in ax.get_yticklabels():
        label.set_fontweight("bold")

    ymin, ymax = ax.get_ylim()
    yrange = ymax - ymin
    tick_h = 0.01 * yrange
    stack_step = 0.05 * yrange
    top_pad = 0.018 * yrange

    for j, condition in enumerate(PLOT_CONDITIONS):
        wt_reps = get_replicates_for_condition(df_plot, "Wild-type", condition)
        group_values = []

        for mutation_type in MUTATION_ORDER:
            group_values.append(get_bar_height(df_plot, mutation_type, PLOT_COND_KEYS[j]))
            group_values.extend(get_replicates_for_condition(df_plot, mutation_type, condition))

        group_values = np.array(group_values, dtype=float)
        group_values = group_values[np.isfinite(group_values)]
        group_top = group_values.max() if group_values.size else ymax
        y_base = group_top + top_pad

        for k, mutation_type in enumerate(MUTATION_ORDER[1:]):
            mutant_reps = get_replicates_for_condition(df_plot, mutation_type, condition)
            pvalue = welch_pvalue(wt_reps, mutant_reps)
            x1 = (x + offsets[0])[j]
            x2 = (x + offsets[k + 1])[j]
            draw_significance_bar(ax, x1, x2, y_base + k * stack_step, tick_h, pvalue)

        ymax = max(ymax, y_base + 2 * stack_step + tick_h * 2)

    if ymax > ax.get_ylim()[1]:
        ax.set_ylim(ax.get_ylim()[0], ymax + 0.02 * (ymax - ax.get_ylim()[0]))

    plt.tight_layout()
    fig.savefig(output_path, bbox_inches="tight")
    plt.close(fig)


# -----------------------------------------------------------------------------
# Main analysis
# -----------------------------------------------------------------------------

def main() -> None:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    merged = merge_allele_files(INPUT_DIR)
    merged.to_csv(OUTPUT_DIR / "merged_ngs_outputs.csv", index=False)

    allele_table = combine_replicate_read_counts(merged)
    allele_table = add_translation_column(allele_table)

    translations = sum_reads_by_translation(allele_table)
    classified = classify_translations(translations)
    classified.to_csv(OUTPUT_DIR / "translated_alleles_classified.csv", index=False)

    type_summary = summarize_by_type(classified)
    type_summary.to_csv(OUTPUT_DIR / "allele_type_summary.csv", index=False)

    condition_columns = {
        "Day 7": ["day_7_a%reads", "day_7_b%reads", "day_7_c%reads"],
        "Day 14": ["day_14_a%reads", "day_14_b%reads", "day_14_c%reads"],
        "Mock": ["mock_a%reads", "mock_b%reads", "mock_c%reads"],
        "1uM Nira": ["nira_a%reads", "nira_b%reads", "nira_c%reads"],
    }

    summary_with_stats = add_condition_stats(type_summary, condition_columns)
    summary_with_log2fc = add_log2_fold_changes(summary_with_stats)

    log2fc_columns = [col for col in summary_with_log2fc.columns if col == "type" or "log2fc" in col]
    df_plot = summary_with_log2fc[log2fc_columns]
    df_plot.to_csv(OUTPUT_DIR / "allele_type_summary_with_log2fc.csv", index=False)

    plot_log2fc(
        df_plot,
        OUTPUT_DIR / "log2fc_average_alleles_with_dots_p_values_exon5.pdf",
        title="Log2FC - exon 5",
    )

    print(f"Analysis complete. Results saved to: {OUTPUT_DIR}")


if __name__ == "__main__":
    main()


/Users/annahoracek/opt/anaconda3/lib/python3.9/site-packages/Bio/Seq.py:2804: BiopythonWarning: Partial codon, len(sequence) not a multiple of three. Explicitly trim the sequence or add trailing N before translation. This may become an error in future.
  warnings.warn(


Analysis complete. Results saved to: /Users/annahoracek/Desktop/Hockemeyer/NGS/2025/MCB153L_reversion_analysis/Exon_5/sample_output
